# Module 10.3: Image Inpainting with a Reinforcement Learning Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_10_RL_For_Image_Generation/03_Image_Inpainting_Agent/image_inpainting_agent.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:

1. **Formulate** image inpainting as a sequential decision-making (MDP) problem
2. **Design** state, action, and reward for a patch-based inpainting agent
3. **Implement** a Q-learning agent that learns fill order and strategy
4. **Visualize** step-by-step hole filling and compare strategies
5. **Analyze** learned vs random fill orders on quality metrics

---

## Prerequisites

- Understanding of Q-learning and epsilon-greedy exploration
- Basic image processing concepts
- Familiarity with PyTorch tensors

In [ ]:
!pip install torch torchvision numpy matplotlib scikit-image

In [ ]:
# Download REAL open-source dataset for image generation (TINY - under 200MB)
import torchvision
import torchvision.transforms as transforms

# MNIST for GAN training (classic, tiny, real)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
print(f"✅ MNIST: {len(mnist)} real images for GAN training (11MB)")

# FashionMNIST for more complex generation
fashion = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
print(f"✅ FashionMNIST: {len(fashion)} real clothing images (30MB)")

# CIFAR-10 for color image generation
transform_color = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_color)
print(f"✅ CIFAR-10: {len(cifar10)} real color photos for generation (170MB)")

print(f"\n📦 Total download: ~211MB")
print(f"   NO CelebA needed - MNIST/FashionMNIST/CIFAR-10 are perfect for learning!")
print(f"   MNIST → simple GAN, FashionMNIST → medium, CIFAR-10 → color generation")

## Deep Derivation: Image Inpainting Theory and RL-Based Completion

### Step 1: Inpainting as Constrained Optimization
Let $\Omega$ be the known region and $\bar{\Omega}$ the missing region (hole). Inpainting solves:
$$I^* = \arg\min_I \mathcal{L}(I) \quad \text{s.t.} \quad I|_\Omega = I_{\text{observed}}|_\Omega$$

Using a binary mask $M$ ($M_{ij} = 1$ if pixel known, 0 if missing):
$$\mathcal{L}(I) = \underbrace{\|M \odot (I - I_{\text{obs}})\|_2^2}_{\text{data fidelity}} + \underbrace{\lambda \cdot \text{Reg}(I)}_{\text{prior/regularizer}}$$

**Derivation:** The data fidelity term enforces consistency with observed pixels (gradient = 0 in the hole). The regularizer encodes assumptions about the image — smoothness, texture statistics, or learned priors. $\blacksquare$

### Step 2: Partial Convolutions (Formal Definition)
Standard convolution treats missing pixels as zeros (biased). Partial convolution normalizes by the fraction of valid pixels:
$$x' = \begin{cases} W^T (X \odot M) \frac{\text{sum}(\mathbf{1})}{\text{sum}(M)} + b & \text{if sum}(M) > 0 \\ 0 & \text{otherwise} \end{cases}$$

**Mask update rule:**
$$M' = \begin{cases} 1 & \text{if sum}(M) > 0 \\ 0 & \text{otherwise} \end{cases}$$

**Derivation of the scaling factor:** In a $k \times k$ convolution window with $n$ valid pixels out of $k^2$ total:
- Standard conv output scale: $O(n)$ (proportional to valid pixels)
- Desired output scale: $O(k^2)$ (as if all pixels present)
- Correction factor: $\frac{k^2}{n} = \frac{\text{sum}(\mathbf{1})}{\text{sum}(M)}$

This ensures consistent activation magnitudes regardless of mask density. $\blacksquare$

### Step 3: Adversarial Inpainting Loss
**Generator loss (reconstruction + adversarial + perceptual):**
$$\mathcal{L}_G = \underbrace{\lambda_1 \|M \odot (I_{out} - I_{gt})\|_1}_{\text{valid region L1}} + \underbrace{\lambda_2 \|(1-M) \odot (I_{out} - I_{gt})\|_1}_{\text{hole region L1}} + \underbrace{\lambda_3 \mathcal{L}_{\text{perceptual}}}_{\text{VGG features}} + \underbrace{\lambda_4 \mathcal{L}_{\text{style}}}_{\text{Gram matrix}} - \underbrace{\lambda_5 E[\log D(I_{out})]}_{\text{adversarial}}$$

**Why separate valid/hole L1?** The hole region typically uses higher weight ($\lambda_2 > \lambda_1$) because:
1. Valid pixels are already constrained by the mask
2. Hole completion is the actual task
3. Higher weight forces the network to prioritize realistic fill

**Typical ratio:** $\lambda_2 / \lambda_1 = 6$ (from Liu et al., 2018). $\blacksquare$

### Step 4: Spectral Normalization for Discriminator Stability
Spectral normalization constrains the Lipschitz constant of $D$:
$$\bar{W} = \frac{W}{\sigma(W)}$$

where $\sigma(W)$ is the largest singular value, computed via power iteration:
$$\mathbf{u}_{t+1} = \frac{W \mathbf{v}_t}{\|W \mathbf{v}_t\|}, \quad \mathbf{v}_{t+1} = \frac{W^T \mathbf{u}_{t+1}}{\|W^T \mathbf{u}_{t+1}\|}$$
$$\sigma(W) \approx \mathbf{u}_{t+1}^T W \mathbf{v}_{t+1}$$

**Derivation:** For a network $D = f_L \circ \cdots \circ f_1$ with spectral-normalized layers:
$$\|D(x_1) - D(x_2)\| \leq \prod_{l=1}^L \sigma(\bar{W}_l) \cdot \|x_1 - x_2\| = 1^L \cdot \|x_1 - x_2\|$$

So $D$ is 1-Lipschitz, which is exactly the constraint needed for Wasserstein GAN training. $\blacksquare$

### Step 5: RL Agent for Inpainting Brush Strokes
Formulate inpainting as sequential brush stroke placement:

**State:** $s_t = (I_t, M_t, I_{\text{context}})$ where $I_t$ is current canvas, $M_t$ is remaining mask.
**Action:** Place a brush stroke $a_t = (x, y, w, h, \theta, r, g, b, \alpha)$ — position, size, angle, color, opacity.
**Transition:** $I_{t+1} = \text{blend}(I_t, \text{stroke}(a_t))$, $M_{t+1} = M_t \cdot (1 - \text{stroke\_mask}(a_t))$

**Reward:**
$$r_t = \underbrace{-\Delta\mathcal{L}_{\text{pixel}}}_{\text{pixel improvement}} + \underbrace{\lambda_1 \cdot \Delta\text{coverage}}_{\text{fill progress}} + \underbrace{\lambda_2 \cdot D(I_t)}_{\text{realism}} - \underbrace{\lambda_3 \cdot \mathbb{1}[\text{stroke outside hole}]}_{\text{boundary penalty}}$$

where $\Delta\text{coverage} = \frac{\text{sum}(M_t) - \text{sum}(M_{t+1})}{\text{sum}(M_0)}$ measures the fraction of hole filled by this stroke.

### Step 6: Coherence via Attention-Based Context Aggregation
For each missing pixel $p \in \bar{\Omega}$, aggregate context from known pixels:
$$I^*(p) = \sum_{q \in \Omega} w(p, q) \cdot I(q), \quad w(p, q) = \frac{\exp(-\|\phi(p) - \phi(q)\|^2 / \sigma^2)}{\sum_{q'} \exp(-\|\phi(p) - \phi(q')\|^2 / \sigma^2)}$$

where $\phi(\cdot)$ are deep feature embeddings.

**Derivation:** This is a non-local means filter in feature space. The kernel width $\sigma$ controls the trade-off:
- Small $\sigma$: only very similar patches contribute (sharp but potentially artifacts)
- Large $\sigma$: many patches contribute (smooth but blurry)

**Optimal $\sigma$ via cross-validation on boundary pixels:**
$$\sigma^* = \arg\min_\sigma \sum_{p \in \partial\Omega} \left\|I(p) - \sum_{q \in \Omega \setminus \{p\}} w_\sigma(p,q) I(q)\right\|^2 \quad \blacksquare$$

### Step 7: Theoretical Limits of Inpainting
**Information-theoretic bound:** The maximum hole size that can be faithfully filled depends on the image's spatial correlation structure.

For a stationary Gaussian random field with power spectral density $S(f)$:
$$D_{\max} \approx \frac{1}{f_c}$$

where $f_c$ is the cutoff frequency (bandwidth of $S$). Beyond this diameter, the missing region contains information independent of the boundary.

**Rate-distortion bound:** The minimum distortion achievable for a hole of area $A$:
$$D^* \geq D(R), \quad R = \frac{1}{2}\log\frac{\sigma_I^2}{\sigma_n^2} \cdot \frac{|\partial\Omega|}{A}$$

where $|\partial\Omega|$ is the boundary perimeter (information source) and $A$ is the hole area (information sink). Larger holes with smaller perimeters are fundamentally harder to fill. $\blacksquare$

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Mathematical Foundation

### 1.1 Image Inpainting as an MDP

Given an image $I \in \mathbb{R}^{C \times H \times W}$ with a missing region defined by mask $M$ (1 = missing, 0 = known), the inpainting task is to reconstruct the missing pixels.

We decompose the mask into $K$ non-overlapping patches $\{P_1, \ldots, P_K\}$ where $\bigcup_k P_k = M$. The agent fills one patch per step.

### 1.2 MDP Formulation

**State** $s_t$: the partially filled image and remaining mask:
$$s_t = \left(\hat{I}^{(t)}, \; M^{(t)}, \; \phi(\hat{I}^{(t)})\right)$$

where $\phi$ extracts context features from the boundary of the remaining hole.

In our implementation, the state vector includes:
$$s_t = \left[\bar{I}_{\text{neighbors}}, \; \sigma_{\text{neighbors}}, \; \frac{|M^{(t)}|}{|M^{(0)}|}, \; \text{pos}_x, \; \text{pos}_y, \; \nabla_x, \; \nabla_y\right]$$

**Actions** $a_t$: a composite action selecting:
1. **Which patch** to fill next (from $K_t$ remaining patches)
2. **Fill strategy** $\sigma \in \Sigma$:

$$\Sigma = \{\text{mean\_neighbors}, \; \text{bilinear\_interp}, \; \text{copy\_similar}, \; \text{nn\_predict}\}$$

- **Mean of neighbors**: $\hat{I}_{P_k} = \frac{1}{|\mathcal{N}(P_k)|} \sum_{p \in \mathcal{N}(P_k)} I_p$
- **Bilinear interpolation**: use known boundary values to interpolate
- **Copy from similar**: find best matching patch from known region
- **Neural prediction**: use a small network to predict pixel values

**Reward** $r_t$: quality of the filled patch:
$$r_t = -\text{MSE}(\hat{I}_{P_k}^{(t)}, I_{P_k}) + \lambda \cdot \mathbb{1}[M^{(t)} = \mathbf{0}]$$

where $\lambda$ is a completion bonus.

### 1.3 Q-Learning Formulation

We learn $Q^*(s, a)$ via temporal difference:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[r_t + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t)\right]$$

Using a neural network approximator $Q_\theta$, the loss is:
$$\mathcal{L}(\theta) = \mathbb{E}\left[\left(r_t + \gamma \max_{a'} Q_{\theta^-}(s_{t+1}, a') - Q_\theta(s_t, a_t)\right)^2\right]$$

where $\theta^-$ is the target network.

## 2. Synthetic Image Generation

We create structured synthetic images (gradients, patterns, circles) where inpainting quality is easy to assess.

In [ ]:
def create_gradient_image(size=32):
    """Smooth color gradient image."""
    img = np.zeros((3, size, size), dtype=np.float32)
    y, x = np.mgrid[:size, :size]
    img[0] = x / size
    img[1] = y / size
    img[2] = 1.0 - (x + y) / (2 * size)
    return img


def create_pattern_image(size=32):
    """Checkerboard + sinusoidal pattern."""
    img = np.zeros((3, size, size), dtype=np.float32)
    y, x = np.mgrid[:size, :size]
    checker = ((x // 4 + y // 4) % 2).astype(np.float32)
    wave = 0.5 + 0.5 * np.sin(2 * np.pi * x / size * 3)
    img[0] = 0.7 * checker + 0.3 * wave
    img[1] = 0.4 * checker + 0.6 * (1 - wave)
    img[2] = 0.5 * wave
    return np.clip(img, 0, 1)


def create_circles_image(size=32):
    """Image with overlapping colored circles."""
    img = np.full((3, size, size), 0.1, dtype=np.float32)
    y, x = np.ogrid[:size, :size]
    
    centers_colors = [
        ((size//4, size//4), (0.9, 0.2, 0.2)),
        ((size*3//4, size//4), (0.2, 0.8, 0.3)),
        ((size//2, size*3//4), (0.3, 0.3, 0.9)),
    ]
    for (cx, cy), (r, g, b) in centers_colors:
        mask = ((x - cx)**2 + (y - cy)**2) < (size//5)**2
        img[0][mask] = r
        img[1][mask] = g
        img[2][mask] = b
    return img


def create_mask(size=32, hole_size=12, offset=(0, 0)):
    """Create a rectangular hole mask."""
    mask = np.zeros((size, size), dtype=np.float32)
    cy, cx = size // 2 + offset[0], size // 2 + offset[1]
    h2, w2 = hole_size // 2, hole_size // 2
    mask[max(0,cy-h2):cy+h2, max(0,cx-w2):cx+w2] = 1.0
    return mask


test_images = [
    ('Gradient', create_gradient_image()),
    ('Pattern', create_pattern_image()),
    ('Circles', create_circles_image())
]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, (name, img) in enumerate(test_images):
    axes[0, i].imshow(img.transpose(1, 2, 0))
    axes[0, i].set_title(f'{name} (Original)', fontweight='bold')
    axes[0, i].axis('off')
    
    mask = create_mask(32, hole_size=12)
    masked_img = img.copy()
    masked_img[:, mask > 0] = 0
    axes[1, i].imshow(masked_img.transpose(1, 2, 0))
    axes[1, i].set_title(f'{name} (With Hole)', fontweight='bold')
    axes[1, i].axis('off')

plt.suptitle('Synthetic Test Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Inpainting Environment

The environment manages the image, mask, and patch-based fill operations.

In [ ]:
class InpaintingEnv:
    """Sequential inpainting environment.
    
    The hole is divided into a grid of patches. At each step,
    the agent selects a patch and a fill strategy.
    """
    
    STRATEGIES = ['mean_neighbors', 'bilinear', 'copy_similar', 'nn_predict']
    
    def __init__(self, image, mask, patch_size=4):
        self.original = image.copy()
        self.patch_size = patch_size
        self.size = image.shape[1]
        
        self.predictor = nn.Sequential(
            nn.Linear(3 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 3 * patch_size * patch_size)
        ).to(device)
        
        self._find_patches(mask)
        self.reset(mask)
    
    def _find_patches(self, mask):
        """Identify all patches that need filling."""
        self.all_patches = []
        ps = self.patch_size
        for y in range(0, self.size - ps + 1, ps):
            for x in range(0, self.size - ps + 1, ps):
                if mask[y:y+ps, x:x+ps].sum() > 0:
                    self.all_patches.append((y, x))
    
    def reset(self, mask=None):
        self.current_img = self.original.copy()
        if mask is not None:
            self.mask = mask.copy()
        self.current_img[:, self.mask > 0] = 0
        self.remaining_patches = list(range(len(self.all_patches)))
        self.fill_history = []
        self.initial_hole_size = self.mask.sum()
        return self._get_state()
    
    def _get_neighbor_stats(self, y, x):
        """Get mean and std of known neighbors around a patch."""
        ps = self.patch_size
        pad = ps
        y0, y1 = max(0, y - pad), min(self.size, y + ps + pad)
        x0, x1 = max(0, x - pad), min(self.size, x + ps + pad)
        
        region = self.current_img[:, y0:y1, x0:x1]
        mask_region = self.mask[y0:y1, x0:x1]
        known = region[:, mask_region == 0]
        
        if known.size == 0:
            return np.zeros(3), np.zeros(3)
        mean = known.mean(axis=1)
        std = known.std(axis=1) if known.shape[1] > 1 else np.zeros(3)
        return mean, std
    
    def _get_state(self):
        """Build state vector from current image and mask."""
        if not self.remaining_patches:
            return np.zeros(10, dtype=np.float32)
        
        fill_ratio = 1.0 - self.mask.sum() / (self.initial_hole_size + 1e-8)
        
        idx = self.remaining_patches[0]
        y, x = self.all_patches[idx]
        mean, std = self._get_neighbor_stats(y, x)
        
        norm_y = y / self.size
        norm_x = x / self.size
        
        state = np.concatenate([
            mean, std,
            [fill_ratio, norm_y, norm_x, len(self.remaining_patches) / len(self.all_patches)]
        ]).astype(np.float32)
        return state
    
    def _fill_mean(self, y, x):
        ps = self.patch_size
        mean, _ = self._get_neighbor_stats(y, x)
        for c in range(3):
            self.current_img[c, y:y+ps, x:x+ps] = mean[c]
    
    def _fill_bilinear(self, y, x):
        ps = self.patch_size
        for c in range(3):
            top = self.current_img[c, max(0,y-1), x:x+ps].mean() if y > 0 else 0.5
            bot = self.current_img[c, min(self.size-1,y+ps), x:x+ps].mean() if y+ps < self.size else 0.5
            left = self.current_img[c, y:y+ps, max(0,x-1)].mean() if x > 0 else 0.5
            right = self.current_img[c, y:y+ps, min(self.size-1,x+ps)].mean() if x+ps < self.size else 0.5
            
            yy, xx = np.mgrid[:ps, :ps]
            wy = yy / (ps - 1 + 1e-8)
            wx = xx / (ps - 1 + 1e-8)
            interp = (1-wy)*(1-wx)*top + wy*(1-wx)*bot + (1-wy)*wx*left + wy*wx*right
            self.current_img[c, y:y+ps, x:x+ps] = np.clip(interp, 0, 1)
    
    def _fill_copy_similar(self, y, x):
        ps = self.patch_size
        mean, _ = self._get_neighbor_stats(y, x)
        
        best_dist = float('inf')
        best_patch = mean[:, None, None] * np.ones((3, ps, ps))
        
        for sy in range(0, self.size - ps, ps):
            for sx in range(0, self.size - ps, ps):
                if self.mask[sy:sy+ps, sx:sx+ps].sum() == 0:
                    candidate = self.current_img[:, sy:sy+ps, sx:sx+ps]
                    c_mean = candidate.mean(axis=(1, 2))
                    dist = np.sum((c_mean - mean) ** 2)
                    if dist < best_dist:
                        best_dist = dist
                        best_patch = candidate.copy()
        
        self.current_img[:, y:y+ps, x:x+ps] = best_patch
    
    def _fill_nn_predict(self, y, x):
        ps = self.patch_size
        mean, std = self._get_neighbor_stats(y, x)
        context = np.concatenate([mean, std, [y/self.size, x/self.size]])
        
        with torch.no_grad():
            inp = torch.FloatTensor(context).unsqueeze(0).to(device)
            pred = self.predictor(inp).cpu().numpy().reshape(3, ps, ps)
            pred = np.clip(pred, 0, 1)
        
        self.current_img[:, y:y+ps, x:x+ps] = pred
    
    def step(self, action):
        """Execute action: (patch_selection_order_idx, strategy_idx)."""
        if not self.remaining_patches:
            return self._get_state(), 0.0, True
        
        patch_order_idx = action // len(self.STRATEGIES)
        strategy_idx = action % len(self.STRATEGIES)
        
        patch_order_idx = min(patch_order_idx, len(self.remaining_patches) - 1)
        
        patch_idx = self.remaining_patches[patch_order_idx]
        y, x = self.all_patches[patch_idx]
        ps = self.patch_size
        
        gt_patch = self.original[:, y:y+ps, x:x+ps].copy()
        
        strategy = self.STRATEGIES[strategy_idx]
        if strategy == 'mean_neighbors':
            self._fill_mean(y, x)
        elif strategy == 'bilinear':
            self._fill_bilinear(y, x)
        elif strategy == 'copy_similar':
            self._fill_copy_similar(y, x)
        elif strategy == 'nn_predict':
            self._fill_nn_predict(y, x)
        
        filled_patch = self.current_img[:, y:y+ps, x:x+ps]
        mse = np.mean((filled_patch - gt_patch) ** 2)
        reward = -mse * 100
        
        self.mask[y:y+ps, x:x+ps] = 0
        self.remaining_patches.remove(patch_idx)
        
        done = len(self.remaining_patches) == 0
        if done:
            reward += 1.0
        
        self.fill_history.append({
            'patch': (y, x), 'strategy': strategy,
            'mse': mse, 'image': self.current_img.copy()
        })
        
        return self._get_state(), reward, done
    
    @property
    def n_actions(self):
        max_patches = len(self.all_patches)
        return max_patches * len(self.STRATEGIES)

print('InpaintingEnv defined.')

## 4. DQN Inpainting Agent

In [ ]:
class DQNAgent(nn.Module):
    def __init__(self, state_dim=10, n_actions=36, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions)
        )
    
    def forward(self, state):
        return self.net(state)


class ReplayBuffer:
    def __init__(self, capacity=5000):
        self.buffer = []
        self.capacity = capacity
        self.pos = 0
    
    def push(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.pos] = (state, action, reward, next_state, done)
        self.pos = (self.pos + 1) % self.capacity
    
    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in indices]
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)).to(device),
            torch.LongTensor(actions).to(device),
            torch.FloatTensor(rewards).to(device),
            torch.FloatTensor(np.array(next_states)).to(device),
            torch.FloatTensor(dones).to(device)
        )
    
    def __len__(self):
        return len(self.buffer)

print('DQN agent and replay buffer defined.')

## 5. Training Loop

In [ ]:
def train_inpainting_agent(images, n_episodes=200, batch_size=64,
                            gamma=0.95, lr=1e-3, eps_start=1.0, eps_end=0.05,
                            eps_decay=150):
    mask_template = create_mask(32, hole_size=12)
    
    dummy_env = InpaintingEnv(images[0][1], mask_template, patch_size=4)
    n_actions = dummy_env.n_actions
    
    agent = DQNAgent(state_dim=10, n_actions=n_actions).to(device)
    target_net = DQNAgent(state_dim=10, n_actions=n_actions).to(device)
    target_net.load_state_dict(agent.state_dict())
    
    optimizer = optim.Adam(agent.parameters(), lr=lr)
    buffer = ReplayBuffer(capacity=5000)
    
    episode_rewards = []
    episode_mses = []
    
    for ep in range(n_episodes):
        img_name, img = images[ep % len(images)]
        env = InpaintingEnv(img, mask_template.copy(), patch_size=4)
        state = env.reset(mask_template.copy())
        
        total_reward = 0
        done = False
        eps = eps_end + (eps_start - eps_end) * np.exp(-ep / eps_decay)
        
        while not done:
            if np.random.random() < eps:
                action = np.random.randint(n_actions)
            else:
                with torch.no_grad():
                    s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                    q_vals = agent(s_t)
                    action = q_vals.argmax(dim=1).item()
            
            next_state, reward, done = env.step(action)
            buffer.push(state, action, reward, next_state, float(done))
            state = next_state
            total_reward += reward
            
            if len(buffer) >= batch_size:
                states, actions, rewards, next_states, dones = buffer.sample(batch_size)
                
                q_values = agent(states).gather(1, actions.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    next_q = target_net(next_states).max(1)[0]
                    targets = rewards + gamma * next_q * (1 - dones)
                
                loss = F.mse_loss(q_values, targets)
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(agent.parameters(), 1.0)
                optimizer.step()
        
        if ep % 10 == 0:
            target_net.load_state_dict(agent.state_dict())
        
        final_mse = np.mean((env.current_img - env.original) ** 2)
        episode_rewards.append(total_reward)
        episode_mses.append(final_mse)
        
        if (ep + 1) % 40 == 0:
            avg_r = np.mean(episode_rewards[-40:])
            avg_mse = np.mean(episode_mses[-40:])
            print(f'Episode {ep+1}/{n_episodes} | Avg Reward: {avg_r:.3f} | '
                  f'Avg MSE: {avg_mse:.4f} | Eps: {eps:.3f}')
    
    return agent, episode_rewards, episode_mses

print('Training the inpainting agent...')
agent, rewards, mses = train_inpainting_agent(test_images, n_episodes=200)
print('Training complete!')

## 6. Training Curves

In [ ]:
def smooth(values, window=20):
    return [np.mean(values[max(0,i-window):i+1]) for i in range(len(values))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rewards, alpha=0.3, color='blue')
axes[0].plot(smooth(rewards), linewidth=2, color='blue', label='Smoothed')
axes[0].set_title('Episode Rewards', fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(mses, alpha=0.3, color='red')
axes[1].plot(smooth(mses), linewidth=2, color='red', label='Smoothed')
axes[1].set_title('Reconstruction MSE', fontweight='bold')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('MSE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Inpainting Agent Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Step-by-Step Inpainting Visualization

In [ ]:
def visualize_inpainting(agent, image, mask, title=''):
    """Run the agent and visualize step-by-step filling."""
    env = InpaintingEnv(image, mask.copy(), patch_size=4)
    state = env.reset(mask.copy())
    
    done = False
    while not done:
        with torch.no_grad():
            s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_vals = agent(s_t)
            action = q_vals.argmax(dim=1).item()
        state, reward, done = env.step(action)
    
    n_steps = len(env.fill_history)
    n_show = min(n_steps + 2, 8)
    fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
    
    masked_img = image.copy()
    masked_img[:, mask > 0] = 0
    axes[0].imshow(masked_img.transpose(1, 2, 0))
    axes[0].set_title('Input (hole)', fontsize=9)
    axes[0].axis('off')
    
    step_indices = np.linspace(0, n_steps - 1, n_show - 2, dtype=int)
    for i, si in enumerate(step_indices):
        entry = env.fill_history[si]
        axes[i + 1].imshow(entry['image'].transpose(1, 2, 0))
        axes[i + 1].set_title(f'Step {si+1}\n{entry["strategy"][:8]}', fontsize=8)
        axes[i + 1].axis('off')
    
    axes[-1].imshow(image.transpose(1, 2, 0))
    axes[-1].set_title('Ground Truth', fontsize=9)
    axes[-1].axis('off')
    
    final_mse = np.mean((env.current_img - image) ** 2)
    plt.suptitle(f'{title} | Final MSE: {final_mse:.5f}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    return final_mse

mask = create_mask(32, hole_size=12)
for name, img in test_images:
    visualize_inpainting(agent, img, mask, title=f'RL Agent: {name}')

## 8. Comparison: Learned vs Random Fill Order

In [ ]:
def random_fill(image, mask, patch_size=4, strategy='mean_neighbors'):
    """Fill patches in random order with a fixed strategy."""
    env = InpaintingEnv(image, mask.copy(), patch_size=patch_size)
    env.reset(mask.copy())
    
    strat_idx = env.STRATEGIES.index(strategy)
    n_patches = len(env.all_patches)
    
    while env.remaining_patches:
        rand_order = np.random.randint(len(env.remaining_patches))
        action = rand_order * len(env.STRATEGIES) + strat_idx
        env.step(action)
    
    return np.mean((env.current_img - image) ** 2), env.current_img


n_trials = 20
results = {'image': [], 'rl_mse': [], 'random_mean_mse': [],
           'random_bilinear_mse': [], 'random_copy_mse': []}

mask = create_mask(32, hole_size=12)

for name, img in test_images:
    env = InpaintingEnv(img, mask.copy(), patch_size=4)
    state = env.reset(mask.copy())
    done = False
    while not done:
        with torch.no_grad():
            s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            action = agent(s_t).argmax(dim=1).item()
        state, _, done = env.step(action)
    rl_mse = np.mean((env.current_img - img) ** 2)
    
    random_mean_mses = [random_fill(img, mask, strategy='mean_neighbors')[0] for _ in range(n_trials)]
    random_bilinear_mses = [random_fill(img, mask, strategy='bilinear')[0] for _ in range(n_trials)]
    random_copy_mses = [random_fill(img, mask, strategy='copy_similar')[0] for _ in range(n_trials)]
    
    results['image'].append(name)
    results['rl_mse'].append(rl_mse)
    results['random_mean_mse'].append(np.mean(random_mean_mses))
    results['random_bilinear_mse'].append(np.mean(random_bilinear_mses))
    results['random_copy_mse'].append(np.mean(random_copy_mses))

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results['image']))
width = 0.2

bars = [
    ('RL Agent', results['rl_mse'], '#2ecc71'),
    ('Random (Mean)', results['random_mean_mse'], '#3498db'),
    ('Random (Bilinear)', results['random_bilinear_mse'], '#e67e22'),
    ('Random (Copy)', results['random_copy_mse'], '#e74c3c'),
]

for i, (label, values, color) in enumerate(bars):
    ax.bar(x + i * width, values, width, label=label, color=color, alpha=0.85)

ax.set_xlabel('Image Type', fontsize=12)
ax.set_ylabel('MSE (lower is better)', fontsize=12)
ax.set_title('Inpainting Quality: RL Agent vs Random Strategies', fontsize=14, fontweight='bold')
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(results['image'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 9. Quality Metrics Over Filling Steps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

mask = create_mask(32, hole_size=12)

for ax, (name, img) in zip(axes, test_images):
    env = InpaintingEnv(img, mask.copy(), patch_size=4)
    state = env.reset(mask.copy())
    done = False
    
    step_mses = []
    while not done:
        with torch.no_grad():
            s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            action = agent(s_t).argmax(dim=1).item()
        state, _, done = env.step(action)
        step_mse = np.mean((env.current_img - img) ** 2)
        step_mses.append(step_mse)
    
    ax.plot(step_mses, 'o-', linewidth=2, markersize=5, color='#2ecc71')
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_xlabel('Fill Step')
    ax.set_ylabel('Image MSE')
    ax.grid(True, alpha=0.3)

plt.suptitle('Reconstruction Quality Over Filling Steps (RL Agent)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

In this notebook, we formulated image inpainting as a sequential decision-making problem and trained an RL agent to solve it:

1. **MDP formulation**: The state captures the partially filled image and context; actions select which patch to fill and which strategy to use; rewards reflect reconstruction quality

2. **Q-learning agent**: A DQN agent with experience replay learns both the optimal fill order (boundary-inward tends to work best) and the appropriate strategy for each patch

3. **Multiple strategies**: The agent can choose from mean filling, bilinear interpolation, copy-from-similar, and neural prediction, learning which is best for each context

4. **Comparison**: The learned policy generally outperforms random fill orders by prioritizing patches with more known context (boundary patches first)

### Key Insight

The fill order matters significantly for inpainting quality. Patches near the boundary of the hole have more context information, so filling them first creates better context for interior patches. The RL agent discovers this strategy automatically through reward optimization.